# Core vs regra simples — Bahia 2024

**Pergunta:** a clusterização `core` (incidência × crescimento) acrescenta algo além de regras epidemiológicas simples?

**Unidade:** município × semana.

**Comparações:**
1. **Regra A:** faixas de incidência (quantis)
2. **Regra B:** nível de incidência + sinal de crescimento
3. **Core (K=4):** clusters remapeados na ordem narrativa

**O que conta como vitória do core:**
- municípios com incidência parecida, mas trajetórias opostas, caem em clusters diferentes
- a origem no core prevê melhor a ida para alta incidência em t+1 do que a regra simples


In [ ]:
from __future__ import annotations

import json
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

warnings.filterwarnings("ignore", category=UserWarning)
sns.set_theme(style="whitegrid", context="notebook")

_cwd = Path.cwd().resolve()
if (_cwd / "ml").is_dir():
    ROOT = _cwd
elif (_cwd.parent / "ml").is_dir():
    ROOT = _cwd.parent
else:
    raise RuntimeError("Abra o notebook a partir de ia-iv/ ou ia-iv/notebooks/.")

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

for name in list(sys.modules):
    if name == "ml" or name.startswith("ml."):
        del sys.modules[name]

from ml.cluster import run_kmeans
from ml.columns import Col, Feat
from ml.config import DEFAULT_K, DEFAULT_YEARS
from ml.dataset import build_features_panel
from ml.paths import region_manifest_path
from ml.preprocess import build_region_parquet
from ml.regions import BA

REGION = BA.slug
ANO = 2024
K = DEFAULT_K
PALETTE = ["#0072B2", "#E69F00", "#009E73", "#D55E00"]
ACCENT = "#0072B2"

print(f"Projeto: {ROOT}")
print(f"Região: {BA.name} | Ano: {ANO} | K={K}")


## 1. Dados e clusters core (ordem narrativa)

Ordem fixa após o K-Means:

| ID | Papel |
|----|--------|
| 0 | baixa incidência |
| 1 | transição: queda |
| 2 | transição: crescimento |
| 3 | alta incidência |


In [ ]:
build_region_parquet(BA, DEFAULT_YEARS)
manifest = json.loads(region_manifest_path(REGION).read_text(encoding="utf-8"))

panel = build_features_panel(REGION, ANO, "core")
res_raw = run_kmeans(panel, "core", k=K)
means_raw = res_raw.cluster_means

id_baixa = int(means_raw[Feat.INCIDENCIA_100K].idxmin())
id_alta = int(means_raw[Feat.INCIDENCIA_100K].idxmax())
middle = [i for i in means_raw.index if i not in (id_baixa, id_alta)]
id_queda = int(means_raw.loc[middle, Feat.CRESCIMENTO].idxmin())
id_cresc = int(means_raw.loc[middle, Feat.CRESCIMENTO].idxmax())
remap = {id_baixa: 0, id_queda: 1, id_cresc: 2, id_alta: 3}

NOMES = {
    0: "baixa incidência",
    1: "transição: queda",
    2: "transição: crescimento",
    3: "alta incidência",
}

df = panel.copy()
df["cluster"] = np.array([remap[int(x)] for x in res_raw.labels])
means = df.groupby("cluster")[[Feat.INCIDENCIA_100K, Feat.CRESCIMENTO, Feat.CASOS]].mean().round(3)
means = means.reindex(range(K))

print(
    f"{manifest['registros_por_ano'].get(str(ANO), '?')} notificações | "
    f"{df[Col.ID_MUNICIP].nunique()} municípios | {len(df):,} linhas"
)
print("Remapeamento:", remap)
display(means.assign(leitura=means.index.map(NOMES)))
display(df["cluster"].value_counts().sort_index().rename("linhas"))


### Regras simples

**Regra A (só incidência):** faixas por quantis da incidência semanal.

Com muitos municípios×semana em zero casos, o `qcut` pode gerar menos de 3 faixas. O código adapta os rótulos.

**Regra B (incidência + sinal):**
- nível: mesma faixa da Regra A
- sinal: crescimento `< 0`, `≈ 0` ou `> 0`

A Regra B já tenta capturar trajetória. A pergunta é se o K-Means ainda separa algo além disso.


In [ ]:
# Regra A: faixas de incidência.
# Com muitos zeros, tercis podem colapsar em menos de 3 faixas; rótulos acompanham o número real de bins.
inc = df[Feat.INCIDENCIA_100K]
bins_A = pd.qcut(inc, q=3, duplicates="drop")
n_A = bins_A.cat.categories.size
labels_A = {
    1: ["baixa"],
    2: ["baixa", "alta"],
    3: ["baixa", "média", "alta"],
}.get(n_A, [f"faixa_{j+1}" for j in range(n_A)])
df["regra_A"] = pd.qcut(inc, q=3, labels=labels_A, duplicates="drop")

# Regra B: nível (mesmas faixas) + sinal de crescimento
nivel = df["regra_A"].astype(str)
sinal = pd.cut(
    df[Feat.CRESCIMENTO],
    bins=[-np.inf, -1e-9, 1e-9, np.inf],
    labels=["queda", "estável", "crescimento"],
)
df["regra_B"] = nivel.astype(str) + " | " + sinal.astype(str)

print(f"Regra A: {n_A} faixa(s) efetiva(s) = {labels_A}")
print("Limites (quantis usados pelo qcut):")
display(df.groupby("regra_A", observed=True)[Feat.INCIDENCIA_100K].agg(["count", "min", "max", "median"]).round(2))
print("\nRegra B (top combinações):")
display(df["regra_B"].value_counts().head(12).to_frame("linhas"))


## 3. Concordância: core vs regras

ARI / NMI medem se os agrupamentos coincidem. Valores baixos sugerem que o core **não** é só uma reescrita da regra.


In [ ]:
# codifica regras como inteiros para métricas
a_codes = df["regra_A"].astype("category").cat.codes
b_codes = df["regra_B"].astype("category").cat.codes
y = df["cluster"].to_numpy()

agree = pd.DataFrame([
    {
        "comparação": "core vs regra A (incidência)",
        "ARI": adjusted_rand_score(y, a_codes),
        "NMI": normalized_mutual_info_score(y, a_codes),
        "n_grupos_regra": int(a_codes.nunique()),
    },
    {
        "comparação": "core vs regra B (incidência + sinal)",
        "ARI": adjusted_rand_score(y, b_codes),
        "NMI": normalized_mutual_info_score(y, b_codes),
        "n_grupos_regra": int(b_codes.nunique()),
    },
])
display(agree.round(3))

ct_a = pd.crosstab(df["regra_A"], df["cluster"], normalize="index")
ct_a.columns = [f"{i}: {NOMES[i]}" for i in ct_a.columns]
display(ct_a.round(3))

fig, ax = plt.subplots(figsize=(8, 3.8))
sns.heatmap(ct_a, annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1, ax=ax)
ax.set_title("Regra A → distribuição nos clusters core")
ax.set_xlabel("cluster core")
ax.set_ylabel("regra A")
plt.tight_layout()


## 4. Onde o core diverge da regra

Caso crítico: **mesma faixa de incidência**, clusters diferentes por trajetória.

Se isso for frequente, o core captura algo que a regra A perde.


In [ ]:
# dentro de cada faixa da regra A, quão misturados estão os clusters?
mix = (
    df.groupby("regra_A")["cluster"]
    .value_counts(normalize=True)
    .rename("pct")
    .reset_index()
)
mix["leitura"] = mix["cluster"].map(NOMES)
display(mix.round(3))

# foco: faixa média da regra A
faixa_media = "A1 média"
sub = df[df["regra_A"] == faixa_media].copy()
print(f"\nLinhas na faixa {faixa_media}: {len(sub):,}")
print("Distribuição de clusters dentro da faixa média:")
display(sub["cluster"].value_counts().sort_index().rename("linhas"))

fig, ax = plt.subplots(figsize=(7.5, 4.2))
colors = sub["cluster"].map({i: PALETTE[i] for i in range(K)})
ax.scatter(sub[Feat.INCIDENCIA_100K], sub[Feat.CRESCIMENTO], c=colors, s=14, alpha=0.45, edgecolors="#333", linewidths=0.2)
ax.axhline(0, color="#666", lw=0.8, ls="--")
ax.set_xlabel("Incidência / 100 mil")
ax.set_ylabel("Crescimento semanal")
ax.set_title(f"Dentro da regra A = {faixa_media}: o core ainda separa trajetórias?")
for i in range(K):
    ax.scatter([], [], c=PALETTE[i], label=f"{i}: {NOMES[i]}")
ax.legend(fontsize=8, frameon=True)
plt.tight_layout()

# quantas linhas da faixa média caem em queda vs crescimento
n_queda = int((sub["cluster"] == 1).sum())
n_cresc = int((sub["cluster"] == 2).sum())
print(
    f"Na faixa média, cluster queda={n_queda:,} | cluster crescimento={n_cresc:,} "
    f"({100 * (n_queda + n_cresc) / max(len(sub), 1):.1f}% das linhas da faixa)"
)


## 5. Utilidade prospectiva: quem vai para alta em t+1?

Comparação posterior (não é treino supervisionado):

- origem = cluster core
- origem = regra A
- origem = regra B (versão colapsada: sinal de crescimento)

Métrica: **P(cluster alta em t+1 | origem em t)** e mediana da incidência em t+1.


In [ ]:
tmp = df[[Col.ID_MUNICIP, Col.SEM_NOT, "cluster", "regra_A", "regra_B", Feat.INCIDENCIA_100K, Feat.CRESCIMENTO]].copy()
tmp = tmp.sort_values([Col.ID_MUNICIP, Col.SEM_NOT])
tmp["cluster_next"] = tmp.groupby(Col.ID_MUNICIP)["cluster"].shift(-1)
tmp["inc_next"] = tmp.groupby(Col.ID_MUNICIP)[Feat.INCIDENCIA_100K].shift(-1)
pairs = tmp.dropna(subset=["cluster_next"]).copy()
pairs["cluster_next"] = pairs["cluster_next"].astype(int)
pairs["foi_alta"] = pairs["cluster_next"] == 3

# Regra B colapsada: só o sinal (mais interpretável na utilidade)
pairs["sinal"] = pd.cut(
    pairs[Feat.CRESCIMENTO],
    bins=[-np.inf, -1e-9, 1e-9, np.inf],
    labels=["queda", "estável", "crescimento"],
)


def util_table(frame: pd.DataFrame, origem_col: str) -> pd.DataFrame:
    rows = []
    for origem, g in frame.groupby(origem_col, observed=True):
        rows.append({
            "origem": origem,
            "n": len(g),
            "P(ir para alta)": g["foi_alta"].mean(),
            "mediana incidência t+1": g["inc_next"].median(),
        })
    out = pd.DataFrame(rows).sort_values("P(ir para alta)", ascending=False)
    return out


util_core = util_table(pairs.assign(origem=pairs["cluster"].map(NOMES)), "origem")
util_a = util_table(pairs, "regra_A")
util_sinal = util_table(pairs, "sinal")

print("Core:")
display(util_core.round(3))
print("Regra A:")
display(util_a.round(3))
print("Só sinal de crescimento:")
display(util_sinal.round(3))

# comparação-chave
p_core_cresc = float(util_core.loc[util_core["origem"] == "transição: crescimento", "P(ir para alta)"].iloc[0])
p_core_baixa = float(util_core.loc[util_core["origem"] == "baixa incidência", "P(ir para alta)"].iloc[0])
p_sinal_cresc = float(util_sinal.loc[util_sinal["origem"] == "crescimento", "P(ir para alta)"].iloc[0])
p_sinal_queda = float(util_sinal.loc[util_sinal["origem"] == "queda", "P(ir para alta)"].iloc[0])
p_a_alta = float(util_a.loc[util_a["origem"] == "A2 alta", "P(ir para alta)"].iloc[0])
p_a_baixa = float(util_a.loc[util_a["origem"] == "A0 baixa", "P(ir para alta)"].iloc[0])

resumo = pd.DataFrame([
    {"método": "core: crescimento", "P(alta t+1)": p_core_cresc},
    {"método": "core: baixa", "P(alta t+1)": p_core_baixa},
    {"método": "sinal: crescimento", "P(alta t+1)": p_sinal_cresc},
    {"método": "sinal: queda", "P(alta t+1)": p_sinal_queda},
    {"método": "regra A: alta", "P(alta t+1)": p_a_alta},
    {"método": "regra A: baixa", "P(alta t+1)": p_a_baixa},
])
display(resumo.round(3))

fig, ax = plt.subplots(figsize=(8, 3.8))
ax.barh(resumo["método"], resumo["P(alta t+1)"], color=ACCENT, alpha=0.85)
ax.set_xlabel("P(ir para cluster alta em t+1)")
ax.set_title("Utilidade prospectiva: core vs regras simples")
plt.tight_layout()

print(
    f"Lift core (crescimento / baixa) = {p_core_cresc / max(p_core_baixa, 1e-9):.2f}x | "
    f"Lift sinal (crescimento / queda) = {p_sinal_cresc / max(p_sinal_queda, 1e-9):.2f}x"
)


## 6. Leitura para a apresentação

Preencha mentalmente com os números acima:

1. Se ARI core vs regra A for baixo e a faixa média misturar queda/crescimento, o core **não é redundante**.
2. Se P(alta | crescimento core) for bem maior que P(alta | baixa), há sinal de utilidade.
3. Se o sinal simples (`Δ > 0`) empatar ou ganhar do core, diga com honestidade: a tendência já carrega a maior parte do valor; o K-Means organiza, mas não inventa magia.

**Formulação segura:**
> O core transforma incidência e crescimento em perfis interpretáveis. A comparação com regras simples testa se esses perfis acrescentam nuance além de limiares fixos.
